# UNIVERSIDAD NACIONAL DE SAN ANTONIO ABAD DEL CUSCO
## FACULTAD DE INGENIERÍA ELÉCTRICA, ELECTRÓNICA, INFORMÁTICA Y MECÁNICA
### ESCUELA PROFESIONAL DE INGENIERÍA INFORMÁTICA Y DE SISTEMAS

---

# INFORME DE GUÍA 02
## “ALGORITMOS DE APROXIMACIÓN APLICADOS A SCHEDULING: LIST SCHEDULING Y LONGEST PROCESSING TIME FIRST”

* **Asignatura:** ALGORITMOS AVANZADOS
* **Semestre Académico:** 2026-II
* **Docente:** Ing. Héctor Eduardo Ugarte Rojas
* **ID:** GRUPO 5
* **Presentado Por:**
  * Choquenaira Quispe, Noe Franklin — `133962`
  * Porroa Sivana, Yeni Ruth — `120893`
  * Quispe Rimachi, Romario — `164257`
  * Yaranga Achahui, Aldo — `103179`
* **Lugar y Fecha:** Cusco-Perú - 2026

---

## 📌 Guía de Uso del Notebook en Google Colab

1. **Abrir en Colab:** Subir este archivo (`guia_2.ipynb`) a Google Colab (`Archivo → Subir notebook` o desde Google Drive).
2. **Ejecutar todo:** Seleccionar en el menú **`Entorno de ejecución → Ejecutar todas`** (`Ctrl + F9`). La ejecución completa toma entre **30 y 50 segundos**.
3. **Descarga de resultados:** La última celda empaqueta y descarga automáticamente un archivo comprimido `resultados_lab02.zip` con todos los CSVs, instancias JSON y figuras PNG generadas.

### Estructura de Secciones del Notebook

| Sección | Descripción y Propósito |
|---|---|
| **0** | Configuración inicial, semilla global reproducible (`SEMILLA_GLOBAL = 2026`) y cronometraje de alta precisión. |
| **1 – 3** | Ejercicios resueltos: Representación con `Counter`, List Scheduling ($O(n \log m)$), LPT y Ramificar-Podar exacto. |
| **4** | Batería de validación y casos límite (36 pruebas unitarias automáticas). |
| **5** | Análisis empírico de complejidad: versión de la guía ($O(n^2/m)$) vs. versión optimizada ($O(n \log m)$). |
| **6** | Generador reproducible de instancias por familias (uniforme, estrecha, sesgada, grande). |
| **7** | Propuesto 1: Evaluación de calidad frente al óptimo ($m \in \{2, 3, 4\}$) y límite del método exacto. |
| **8** | Propuesto 2: Efecto del orden, sensibilidad según razón $n/m$ y escalabilidad asintótica ($n$ hasta $100\,000$). |
| **9** | Exportación y descarga automática del paquete de evidencias. |

> **Nota metodológica:** Todos los makespans, cotas, nodos explorados y clasificaciones son **estrictamente deterministas** gracias a la semilla maestra. Los tiempos de cómputo dependen de la CPU de la máquina virtual de Google Colab pero preservan sus tendencias asintóticas.



---
## 0. Configuración y utilidades de medición

- `medir_mediana` cronometra solo la llamada al algoritmo (los datos ya están generados),
  repite la medición y devuelve la **mediana**. Si una llamada dura microsegundos, la repite
  varias veces dentro de cada medición y divide, como hace `timeit`.
- `SEMILLA_GLOBAL = 2026` hace reproducibles todas las instancias y permutaciones.
- Colores fijos por algoritmo en todos los gráficos: **List Scheduling azul**, **LPT naranja**,
  **Ramificar-Podar aqua**, cada uno con su propio marcador.

In [ ]:
# ============================================================================
# Celda 0 - Configuracion, rutas y utilidades de medicion
# ============================================================================
import csv
import heapq
import json
import math
import os
import platform
import shutil
import sys
from collections import Counter
from itertools import permutations
from math import ceil
from random import Random
from statistics import mean, median
from time import perf_counter

import matplotlib
import matplotlib.pyplot as plt

# Deteccion del entorno: en Colab los archivos se escriben en /content, el
# disco temporal de la sesion. Fuera de Colab se usa una carpeta local.
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# pandas solo se usa para mostrar las tablas con mejor formato. Si no esta
# disponible, las tablas se imprimen como texto alineado.
try:
    import pandas as pd
    from IPython.display import display
    HAY_PANDAS = True
except ImportError:
    HAY_PANDAS = False

CARPETA_BASE = "/content/laboratorio02" if EN_COLAB else "laboratorio02"
CARPETA_RESULTADOS = os.path.join(CARPETA_BASE, "resultados")
CARPETA_FIGURAS = os.path.join(CARPETA_BASE, "figuras")
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
os.makedirs(CARPETA_FIGURAS, exist_ok=True)

# Semilla unica para TODO el laboratorio: cualquier equipo que ejecute el
# notebook genera exactamente las mismas instancias y permutaciones.
SEMILLA_GLOBAL = 2026

# Repeticiones para medir tiempos (se reporta la mediana).
REPETICIONES = 7

# ---------------------------------------------------------------------------
# Estilo de los graficos: marcas finas, cuadricula tenue y solida, sin bordes
# superiores ni derechos. Colores fijos por algoritmo (nunca por posicion):
#   List Scheduling -> azul, LPT -> naranja, Ramificar-Podar / optimo -> aqua.
# La paleta fue validada para daltonismo; como el aqua tiene poco contraste,
# cada serie lleva ademas un marcador distinto y leyenda.
# ---------------------------------------------------------------------------
COLOR = {"LS": "#2a78d6", "LPT": "#eb6834", "BB": "#1baf7a"}
MARCADOR = {"LS": "o", "LPT": "s", "BB": "^"}
TINTA = "#0b0b0b"
TINTA_SECUNDARIA = "#52514e"
TINTA_TENUE = "#898781"
CUADRICULA = "#e1e0d9"
EJE = "#c3c2b7"

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": EJE,
    "axes.linewidth": 0.8,
    "axes.labelcolor": TINTA_SECUNDARIA,
    "axes.titlecolor": TINTA,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": CUADRICULA,
    "grid.linewidth": 0.8,
    "grid.linestyle": "-",
    "xtick.color": TINTA_TENUE,
    "ytick.color": TINTA_TENUE,
    "xtick.labelcolor": TINTA_SECUNDARIA,
    "ytick.labelcolor": TINTA_SECUNDARIA,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "font.size": 10,
})


def medir_mediana(funcion, *argumentos, repeticiones=REPETICIONES):
    # Ejecuta la funcion varias veces y devuelve (resultado, mediana en s).
    #
    # * El cronometro rodea UNICAMENTE la llamada al algoritmo: los argumentos
    #   ya deben estar construidos, asi que generar datos nunca se mide.
    # * La mediana es robusta frente a interrupciones del sistema operativo.
    # * Llamadas muy cortas (microsegundos) se repiten "lazos" veces dentro de
    #   cada medicion y se divide, igual que hace timeit: una sola llamada de
    #   pocos microsegundos queda dominada por el ruido del reloj.
    inicio = perf_counter()
    resultado = funcion(*argumentos)             # calibracion y calentamiento
    unica = perf_counter() - inicio
    lazos = 1 if unica >= 2e-3 else min(1000, max(1, int(2e-3 / max(unica, 1e-7))))

    tiempos = []
    for _ in range(repeticiones):
        inicio = perf_counter()
        for _ in range(lazos):
            resultado = funcion(*argumentos)
        tiempos.append((perf_counter() - inicio) / lazos)
    return resultado, median(tiempos)


def formato_tiempo(segundos):
    # Formatea un tiempo en segundos con una unidad legible.
    if segundos != segundos:                 # NaN: ejecucion abortada
        return "abortado"
    if segundos < 1e-3:
        return "%.1f us" % (segundos * 1e6)
    if segundos < 1.0:
        return "%.2f ms" % (segundos * 1e3)
    return "%.3f s" % segundos


def mostrar_tabla(encabezados, filas, titulo=None, max_filas=None):
    # Muestra una tabla: con pandas si esta disponible, en texto si no.
    if titulo:
        print(titulo)
    if max_filas is not None and len(filas) > max_filas:
        print("(se muestran %d de %d filas; el CSV contiene todas)" % (max_filas, len(filas)))
        filas = filas[:max_filas]
    if HAY_PANDAS:
        display(pd.DataFrame(filas, columns=encabezados))
        return
    anchos = [max(len(str(encabezados[i])),
                  max((len(str(f[i])) for f in filas), default=0)) + 2
              for i in range(len(encabezados))]
    print("".join(str(encabezados[i]).ljust(anchos[i]) for i in range(len(encabezados))))
    print("-" * sum(anchos))
    for fila in filas:
        print("".join(str(fila[i]).ljust(anchos[i]) for i in range(len(fila))))


def guardar_csv(nombre_archivo, encabezados, filas):
    # Conserva los datos crudos: permite rehacer tablas y graficos sin
    # volver a ejecutar los experimentos.
    ruta = os.path.join(CARPETA_RESULTADOS, nombre_archivo)
    with open(ruta, "w", newline="", encoding="utf-8") as archivo:
        escritor = csv.writer(archivo)
        escritor.writerow(encabezados)
        escritor.writerows(filas)
    print("Guardado: " + ruta)
    return ruta


def guardar_figura(figura, nombre_archivo):
    # Guarda la figura en PNG (para el informe) y la muestra en el notebook.
    ruta = os.path.join(CARPETA_FIGURAS, nombre_archivo)
    figura.savefig(ruta, bbox_inches="tight")
    print("Guardado: " + ruta)
    plt.show()


def imprimir_entorno():
    # El tiempo observado depende del procesador, del interprete y del sistema
    # operativo: el entorno forma parte del reporte experimental.
    print("Entorno de ejecucion")
    print("  Colab            :", EN_COLAB)
    print("  Python           :", sys.version.split()[0], platform.python_implementation())
    print("  Sistema          :", platform.system(), platform.release())
    print("  Arquitectura     :", platform.machine())
    print("  Carpeta de salida:", CARPETA_BASE)


imprimir_entorno()

---
## 1. Ejercicio resuelto 1 — representación y validación

**Problema $P_m \parallel C_{\max}$.** Hay $n$ trabajos independientes con tiempos $p_j > 0$ y
$m$ máquinas idénticas. Cada trabajo va a una sola máquina y no se interrumpe. Si $L_i$ es
la carga de la máquina $i$, se minimiza el **makespan** $C_{\max} = \max_i L_i$.

**Cota inferior.** Para tiempos enteros:
$$LB = \max\left\{\left\lceil \frac{\sum_j p_j}{m} \right\rceil,\; \max_j p_j\right\} \le \mathrm{OPT}.$$

**Representación.** Una planificación es una lista de $m$ listas; la lista $i$ contiene los
trabajos de la máquina $i$. La factibilidad se verifica con `Counter` (multiconjunto) porque
puede haber trabajos con la misma duración: `set()` perdería las repeticiones.

In [ ]:
# ============================================================================
# Celda 1 - Ejercicio resuelto 1: representacion y validacion
# ============================================================================
# Representacion adoptada en todo el laboratorio:
#   * trabajos      : lista de enteros positivos p_j (puede haber repetidos);
#   * m             : numero de maquinas identicas (entero >= 1);
#   * planificacion : lista de m listas; la lista i contiene los trabajos
#                     asignados a la maquina i.


def validar_instancia(trabajos, m):
    # Rechaza entradas que no pertenecen al problema Pm || Cmax.
    # El marco teorico exige m >= 1 y tiempos p_j > 0 enteros.
    if not isinstance(m, int) or m <= 0:
        raise ValueError("El numero de maquinas debe ser un entero positivo")
    for p in trabajos:
        if not isinstance(p, int) or p <= 0:
            raise ValueError("Cada tiempo de proceso debe ser un entero positivo: %r" % (p,))


def cargas(planificacion):
    # L_i = suma de los tiempos asignados a la maquina i.
    return [sum(maquina) for maquina in planificacion]


def makespan(planificacion):
    # Cmax = maxima carga. default=0 cubre el caso sin maquinas/trabajos.
    return max(cargas(planificacion), default=0)


def es_factible(trabajos, planificacion, m):
    # Una planificacion es factible si:
    #   1) tiene exactamente m maquinas;
    #   2) cada trabajo aparece exactamente una vez (ni falta ni sobra).
    # Se compara con Counter (multiconjunto) y NO con set: una instancia puede
    # tener trabajos con la misma duracion, y set() perderia las repeticiones.
    if len(planificacion) != m:
        return False
    asignados = [p for maquina in planificacion for p in maquina]
    return Counter(asignados) == Counter(trabajos)


def cota_inferior(trabajos, m):
    # LB = max( ceil(sum p_j / m), max p_j ) <= OPT
    #   * carga promedio: alguna maquina recibe al menos el promedio;
    #   * trabajo mas largo: no se interrumpe, cabe entero en una maquina.
    # Se usa aritmetica entera (-(-a // b) == ceil(a / b)) para evitar errores
    # de redondeo de punto flotante con sumas grandes.
    if m <= 0:
        raise ValueError("El numero de maquinas debe ser positivo")
    if not trabajos:
        return 0
    return max(-(-sum(trabajos) // m), max(trabajos))


# --- Ejemplo de la guia ------------------------------------------------------
trabajos = [7, 6, 5, 4, 3]
planificacion = [[7], [6, 3], [5, 4]]

print("Factible     :", es_factible(trabajos, planificacion, 3))
print("Cargas       :", cargas(planificacion))
print("Makespan     :", makespan(planificacion))
print("Cota inferior:", cota_inferior(trabajos, 3))

# Si el makespan alcanza una cota inferior valida, la planificacion es optima:
# ninguna planificacion puede bajar de LB, y esta ya la alcanzo.
if makespan(planificacion) == cota_inferior(trabajos, 3):
    print("-> El makespan alcanza la cota inferior: la planificacion es OPTIMA.")

---
## 2. Ejercicio resuelto 2 — List Scheduling y LPT

**List Scheduling (LS)** recorre los trabajos en el orden recibido y asigna cada uno a la
máquina de menor carga. Con una cola de prioridad mínima de tuplas `(carga, id)`, cada
asignación cuesta $O(\log m)$ y el total es $O(n \log m)$. Garantía (Graham, 1966):
$$C_{LS} \le \left(2 - \frac{1}{m}\right)\mathrm{OPT}.$$

**LPT** ordena de mayor a menor y aplica LS: $O(n \log n)$ por el ordenamiento. Garantía
(Graham, 1969):
$$C_{LPT} \le \left(\frac{4}{3} - \frac{1}{3m}\right)\mathrm{OPT}.$$

**Desempate.** Con cargas iguales gana la máquina de menor identificador: la salida es
determinista.

> **Cambio respecto de la guía.** La versión de la guía guarda la lista de trabajos dentro
> del heap y la copia en cada paso (`asignados + [trabajo]`), lo que cuesta $O(n^2/m)$ en
> total. Aquí el heap guarda solo `(carga, id)` y las listas viven aparte. La sección 5
> demuestra que ambas producen **exactamente** la misma planificación y mide la diferencia.

In [ ]:
# ============================================================================
# Celda 2 - Ejercicio resuelto 2: List Scheduling y LPT con cola de prioridad
# ============================================================================


def list_scheduling(trabajos, m):
    # Procesa los trabajos en el ORDEN RECIBIDO y asigna cada uno a la maquina
    # de menor carga actual. Devuelve (planificacion, makespan).
    #
    # Cola de prioridad minima con tuplas (carga, identificador):
    #   * heap[0] es siempre la maquina de menor carga -> consulta O(1);
    #   * actualizarla cuesta O(log m)  ->  costo total O(n log m).
    #
    # Regla de desempate: con cargas iguales, Python compara el segundo
    # elemento de la tupla, asi que gana la maquina de MENOR identificador.
    # La eleccion es determinista: dos ejecuciones dan la misma planificacion.
    #
    # Diferencia con la version de la guia: alli cada entrada del heap guarda
    # su lista de trabajos y se copia en cada paso (asignados + [trabajo]).
    # Esa copia cuesta O(|asignados|) y lleva el total a O(n^2 / m) en listas
    # largas. Aqui el heap guarda solo (carga, id) y la lista de cada maquina
    # vive aparte, con append O(1). La asignacion producida es la misma.
    if m <= 0:
        raise ValueError("El numero de maquinas debe ser positivo")

    planificacion = [[] for _ in range(m)]
    heap = [(0, i) for i in range(m)]        # todas vacias: ya es un heap valido
    heapq.heapify(heap)

    for trabajo in trabajos:
        carga, maquina = heap[0]             # maquina menos cargada
        planificacion[maquina].append(trabajo)
        # heapreplace = extraer el minimo e insertar el nuevo en una sola
        # operacion O(log m), mas eficiente que heappop + heappush.
        heapq.heapreplace(heap, (carga + trabajo, maquina))

    return planificacion, max(carga for carga, _ in heap)


def lpt(trabajos, m):
    # Longest Processing Time first: ordenar de MAYOR a MENOR y aplicar
    # List Scheduling. Costo O(n log n) del ordenamiento + O(n log m).
    # sorted() crea una copia: la lista original del usuario no se modifica.
    ordenados = sorted(trabajos, reverse=True)
    return list_scheduling(ordenados, m)


# --- Ejemplo de la guia ------------------------------------------------------
trabajos = [3, 7, 4, 6, 5]
plan_ls, valor_ls = list_scheduling(trabajos, 3)
plan_lpt, valor_lpt = lpt(trabajos, 3)

print("List Scheduling:", plan_ls, "-> makespan", valor_ls)
print("LPT            :", plan_lpt, "-> makespan", valor_lpt)
print("Cota inferior  :", cota_inferior(trabajos, 3))

# Ambos alcanzan la cota inferior (9), luego ambos son optimos EN ESTA
# INSTANCIA. Eso no los convierte en algoritmos exactos: la garantia solo
# asegura Cmax <= rho * OPT en cualquier instancia.
assert es_factible(trabajos, plan_ls, 3) and es_factible(trabajos, plan_lpt, 3)

---
## 3. Ejercicio resuelto 3 — Ramificar-Podar exacto

El nivel $k$ del árbol decide en qué máquina va el $k$-ésimo trabajo (ordenados de mayor a menor).

| Elemento | En este algoritmo |
|---|---|
| **Incumbente** | se inicia con la solución de LPT: es buena y permite podar desde el primer nivel |
| **Cota del nodo** | $\max(\text{carga máxima actual},\ \lceil \sum p / m \rceil,\ \text{trabajo restante más largo})$ |
| **Poda** | si la cota del nodo $\ge$ incumbente, la rama no puede mejorar |
| **Simetría** | máquinas con la misma carga son intercambiables: se prueba solo una |
| **Certificado** | si LPT ya alcanza la cota global, es óptima y no se explora ningún nodo |

El peor caso sigue siendo exponencial ($m^n$ hojas); la sección 7 mide su límite práctico.

In [ ]:
# ============================================================================
# Celda 3 - Ejercicio resuelto 3: Ramificar-Podar exacto
# ============================================================================


class LimiteNodosExcedido(Exception):
    # Tope de seguridad para no bloquear la sesion con instancias grandes:
    # el peor caso del metodo exacto sigue siendo exponencial (m^n hojas).
    def __init__(self, nodos):
        Exception.__init__(self, "Limite de nodos excedido (%d nodos)" % nodos)
        self.nodos = nodos


def branch_and_bound(trabajos, m, limite_nodos=None):
    # Calcula una planificacion OPTIMA. Devuelve (plan, optimo, nodos).
    #
    # Arbol de busqueda: el nivel k decide en que maquina va el k-esimo
    # trabajo (ordenados de mayor a menor). Elementos del metodo:
    #   * incumbente : la mejor solucion conocida; se inicia con LPT, que ya
    #                  es buena y permite podar desde el primer nivel;
    #   * cota       : LB del nodo = max(carga maxima actual,
    #                                    ceil(carga total / m),
    #                                    trabajo restante mas largo);
    #   * poda       : si LB del nodo >= incumbente, la rama no puede mejorar.
    #
    # nodos = llamadas a buscar(); es la metrica de trabajo del metodo exacto.
    if m <= 0:
        raise ValueError("El numero de maquinas debe ser positivo")

    trabajos = sorted(trabajos, reverse=True)
    n = len(trabajos)
    mejor_plan, mejor_valor = lpt(trabajos, m)
    cargas_actuales = [0] * m
    plan_actual = [[] for _ in range(m)]
    nodos = 0

    # La carga total no cambia al repartir: ceil(total / m) es constante.
    # (La guia la recalcula en cada nodo con sum(); el valor es el mismo.)
    cota_promedio = -(-sum(trabajos) // m)

    def buscar(k):
        nonlocal mejor_plan, mejor_valor, nodos
        nodos += 1
        if limite_nodos is not None and nodos > limite_nodos:
            raise LimiteNodosExcedido(nodos)

        if k == n:                                   # hoja: todo asignado
            valor = max(cargas_actuales, default=0)
            if valor < mejor_valor:                  # nuevo incumbente
                mejor_valor = valor
                mejor_plan = [x.copy() for x in plan_actual]
            return

        trabajo = trabajos[k]
        cargas_probadas = set()

        for i in range(m):
            # Ruptura de simetria: dos maquinas con la misma carga son
            # intercambiables; basta probar una de ellas.
            if cargas_actuales[i] in cargas_probadas:
                continue
            cargas_probadas.add(cargas_actuales[i])

            nueva_carga = cargas_actuales[i] + trabajo
            if nueva_carga >= mejor_valor:           # poda directa
                continue

            cargas_actuales[i] = nueva_carga
            plan_actual[i].append(trabajo)

            # Como los trabajos estan ordenados de mayor a menor, el restante
            # mas largo es trabajos[k + 1].
            restante_mas_largo = trabajos[k + 1] if k + 1 < n else 0
            lb = max(max(cargas_actuales), cota_promedio, restante_mas_largo)
            if lb < mejor_valor:
                buscar(k + 1)

            plan_actual[i].pop()                     # deshacer (backtracking)
            cargas_actuales[i] -= trabajo

            # Si se probo una maquina vacia, las demas vacias son simetricas.
            if nueva_carga == trabajo:
                break

    # Si LPT ya alcanza la cota global, es optima y no hace falta buscar:
    # este es el argumento de "alcanzar una cota inferior valida".
    if mejor_valor == cota_inferior(trabajos, m):
        return mejor_plan, mejor_valor, nodos

    buscar(0)
    return mejor_plan, mejor_valor, nodos


# --- Ejemplo de la guia ------------------------------------------------------
plan, optimo, nodos = branch_and_bound([7, 6, 5, 4, 3], 3)
print("Plan optimo :", plan)
print("Optimo      :", optimo)
print("Nodos       :", nodos, "(0 = certificado por la cota sin explorar el arbol)")

# Instancia donde LPT NO es optimo: el metodo exacto si debe explorar.
plan, optimo, nodos = branch_and_bound([3, 3, 2, 2, 2], 2)
print("\n[3,3,2,2,2], m=2 -> LPT =", lpt([3, 3, 2, 2, 2], 2)[1],
      "| optimo =", optimo, "| plan =", plan, "| nodos =", nodos)

---
## 4. Pruebas de factibilidad y casos límite

Cada prueba compara lo obtenido con lo esperado y **detiene el notebook si falla**. Grupos:

1. **Validación de planificaciones:** trabajo faltante, duplicado, ajeno, número de máquinas
   incorrecto y el caso de repetidos donde `set()` aceptaría una planificación inválida.
2. **Entradas inválidas:** $m \le 0$, tiempos nulos, negativos o no enteros.
3. **Casos límite de tamaño:** sin trabajos, $m = 1$, $m > n$, $n = m$, todos iguales y un caso
   con $LB < \mathrm{OPT}$ (la cota no siempre se alcanza).
4. **Instancias ajustadas:** alcanzan **exactamente** la garantía teórica.
   - Para LS: $m(m-1)$ trabajos de duración 1 y uno de duración $m$ → $C_{LS} = 2m - 1$, $\mathrm{OPT} = m$.
   - Para LPT: dos trabajos de cada duración $2m-1, \dots, m+1$ y tres de duración $m$ →
     $C_{LPT} = 4m - 1$, $\mathrm{OPT} = 3m$.
5. **Propiedades de la implementación:** no se modifica la entrada, el desempate es
   determinista y el tope de nodos detiene la búsqueda.
6. **Factibilidad masiva:** 300 instancias aleatorias, 900 planificaciones validadas.

In [ ]:
# ============================================================================
# Celda 4 - Pruebas de factibilidad y casos limite
# ============================================================================
# Cada prueba compara el resultado obtenido con el esperado. Si alguna falla,
# la celda se detiene con AssertionError: el notebook no sigue con algoritmos
# incorrectos.

resultados_pruebas = []


def registrar(nombre, obtenido, esperado):
    # Anota una prueba y detiene la ejecucion si no se cumple.
    ok = obtenido == esperado
    resultados_pruebas.append([len(resultados_pruebas) + 1, nombre, str(obtenido),
                               str(esperado), "OK" if ok else "FALLA"])
    assert ok, "Fallo la prueba '%s': obtenido %r, esperado %r" % (nombre, obtenido, esperado)


def lanza_error(funcion, *argumentos):
    # True si la funcion rechaza la entrada con ValueError.
    try:
        funcion(*argumentos)
    except ValueError:
        return True
    return False


def verificar_solucion(trabajos, m, plan, valor):
    # Toda salida de LS, LPT o Ramificar-Podar debe ser factible, su makespan
    # debe coincidir con el valor devuelto y no puede estar por debajo de LB.
    return (es_factible(trabajos, plan, m)
            and makespan(plan) == valor
            and valor >= cota_inferior(trabajos, m))


def instancia_ajustada_ls(m):
    # Instancia clasica de Graham: m(m-1) trabajos de duracion 1 seguidos de
    # uno de duracion m. LS reparte los unos parejo (carga m-1 en cada
    # maquina) y el trabajo largo llega al final: Cmax = 2m - 1, OPT = m.
    return [1] * (m * (m - 1)) + [m]


def instancia_ajustada_lpt(m):
    # Instancia clasica de LPT: dos trabajos de cada duracion 2m-1, ..., m+1
    # y tres de duracion m (2m + 1 trabajos). LPT = 4m - 1, OPT = 3m.
    trabajos = []
    for p in range(2 * m - 1, m, -1):
        trabajos += [p, p]
    return trabajos + [m, m, m]


# ---- 1. Validacion de planificaciones --------------------------------------
t = [7, 6, 5, 4, 3]
registrar("Ejemplo de la guia es factible", es_factible(t, [[7], [6, 3], [5, 4]], 3), True)
registrar("Cargas del ejemplo", cargas([[7], [6, 3], [5, 4]]), [7, 9, 9])
registrar("Falta un trabajo -> no factible", es_factible(t, [[7], [6, 3], [5]], 3), False)
registrar("Trabajo duplicado -> no factible", es_factible(t, [[7, 3], [6, 3], [5, 4]], 3), False)
registrar("Numero de maquinas incorrecto", es_factible(t, [[7, 6], [5, 4, 3]], 3), False)
registrar("Trabajo ajeno a la instancia", es_factible(t, [[7], [6, 3], [5, 8]], 3), False)
# Multiplicidades: con set() esta planificacion pasaria como valida, porque
# {5, 3} == {5, 5, 3}; Counter detecta que falta un 5.
registrar("Repetidos: set() aceptaria, Counter no",
          (set([5, 3]) == set([5, 5, 3]), es_factible([5, 5, 3], [[5], [3]], 2)), (True, False))
registrar("Repetidos bien asignados", es_factible([5, 5, 3], [[5], [5, 3]], 2), True)

# ---- 2. Entradas invalidas -------------------------------------------------
registrar("m = 0 en cota_inferior -> ValueError", lanza_error(cota_inferior, [1, 2], 0), True)
registrar("m < 0 en list_scheduling -> ValueError", lanza_error(list_scheduling, [1, 2], -1), True)
registrar("m = 0 en Ramificar-Podar -> ValueError", lanza_error(branch_and_bound, [1], 0), True)
registrar("Tiempo p_j = 0 -> ValueError", lanza_error(validar_instancia, [3, 0, 2], 2), True)
registrar("Tiempo negativo -> ValueError", lanza_error(validar_instancia, [3, -1], 2), True)
registrar("Tiempo no entero -> ValueError", lanza_error(validar_instancia, [2.5, 1], 2), True)

# ---- 3. Casos limite de tamano ---------------------------------------------
registrar("Sin trabajos: LB = 0", cota_inferior([], 3), 0)
registrar("Sin trabajos: LS da 3 maquinas vacias", list_scheduling([], 3), ([[], [], []], 0))
registrar("Sin trabajos: optimo = 0", branch_and_bound([], 3)[1], 0)

t = [4, 9, 2, 7]
registrar("m = 1: LS = suma", list_scheduling(t, 1)[1], sum(t))
registrar("m = 1: optimo = suma", branch_and_bound(t, 1)[1], sum(t))
registrar("m > n: LS = trabajo mas largo", list_scheduling(t, 6)[1], max(t))
registrar("m > n: LB = OPT = trabajo mas largo",
          (cota_inferior(t, 6), branch_and_bound(t, 6)[1]), (9, 9))
registrar("n = m: un trabajo por maquina", lpt(t, 4)[1], 9)
registrar("Todos iguales, n = 3m: OPT = 3p = LB",
          (branch_and_bound([5] * 9, 3)[1], cota_inferior([5] * 9, 3)), (15, 15))
# LB no siempre es alcanzable: 3 trabajos de 2 en 2 maquinas -> LB = 3, OPT = 4.
registrar("LB < OPT: [2,2,2], m=2", (cota_inferior([2, 2, 2], 2), branch_and_bound([2, 2, 2], 2)[1]), (3, 4))

# ---- 4. Instancias ajustadas: la garantia se alcanza pero no se supera -----
for m in (2, 3, 4):
    t = instancia_ajustada_ls(m)
    _, c_ls = list_scheduling(t, m)
    _, opt, _ = branch_and_bound(t, m)
    registrar("Ajustada LS m=%d: r = 2 - 1/m" % m, (c_ls, opt, c_ls * m == (2 * m - 1) * opt),
              (2 * m - 1, m, True))
for m in (2, 3, 4):
    t = instancia_ajustada_lpt(m)
    _, c_lpt = lpt(t, m)
    _, opt, _ = branch_and_bound(t, m)
    registrar("Ajustada LPT m=%d: r = 4/3 - 1/(3m)" % m, (c_lpt, opt), (4 * m - 1, 3 * m))

# ---- 5. Propiedades de las implementaciones --------------------------------
t = [8, 3, 5, 3, 9, 1]
copia = list(t)
lpt(t, 2)
list_scheduling(t, 2)
registrar("LS y LPT no modifican la entrada", t, copia)
registrar("Desempate determinista (dos ejecuciones)", list_scheduling(t, 3), list_scheduling(t, 3))
registrar("Desempate: con cargas iguales gana el menor id", list_scheduling([5, 5, 5], 3)[0],
          [[5], [5], [5]])


def aborta_por_limite():
    # La instancia ajustada de LPT obliga a explorar (LPT = 15 > LB = 12);
    # con un tope de 1 nodo la busqueda debe abortar de forma controlada.
    try:
        branch_and_bound(instancia_ajustada_lpt(4), 4, limite_nodos=1)
    except LimiteNodosExcedido:
        return True
    return False


registrar("Limite de nodos detiene la busqueda", aborta_por_limite(), True)

# ---- 6. Factibilidad masiva: 300 instancias aleatorias pequenas ------------
generador = Random(SEMILLA_GLOBAL)
salidas_validas = 0
orden_correcto = 0
for _ in range(300):
    m = generador.randint(1, 4)
    t = [generador.randint(1, 20) for _ in range(generador.randint(0, 9))]
    plan_ls, v_ls = list_scheduling(t, m)
    plan_lpt, v_lpt = lpt(t, m)
    plan_bb, v_bb, _ = branch_and_bound(t, m)
    if all(verificar_solucion(t, m, p, v) for p, v in
           ((plan_ls, v_ls), (plan_lpt, v_lpt), (plan_bb, v_bb))):
        salidas_validas += 1
    # El optimo nunca puede superar a una heuristica: OPT <= LPT y OPT <= LS.
    if v_bb <= v_lpt and v_bb <= v_ls:
        orden_correcto += 1
registrar("300 instancias: las 900 salidas son factibles", salidas_validas, 300)
registrar("300 instancias: OPT <= LPT y OPT <= LS", orden_correcto, 300)

mostrar_tabla(["#", "prueba", "obtenido", "esperado", "estado"], resultados_pruebas,
              "Pruebas de factibilidad y casos limite")
print("\n%d de %d pruebas superadas." % (
    sum(1 for f in resultados_pruebas if f[4] == "OK"), len(resultados_pruebas)))
guardar_csv("pruebas_factibilidad.csv", ["n", "prueba", "obtenido", "esperado", "estado"],
            resultados_pruebas)

---
## 5. Complejidad de List Scheduling: versión de la guía vs. optimizada

Copiar una lista de $k$ elementos cuesta $O(k)$. En la versión de la guía, una máquina que
termina con $n/m$ trabajos paga $1 + 2 + \dots + n/m = O\big((n/m)^2\big)$ copias; con $m$
máquinas el total es $O(n^2/m)$. La versión optimizada es $O(n \log m)$.

La celda comprueba primero que **ambas producen la misma salida** en 500 instancias y
después mide el tiempo con $m = 4$ multiplicando $n$ por 4 en cada fila: una función lineal
crece ×4 y una cuadrática ×16.

In [ ]:
# ============================================================================
# Celda 5 - Complejidad de List Scheduling: version de la guia vs optimizada
# ============================================================================
# La version de la guia guarda la lista de trabajos dentro de la tupla del heap
# y la COPIA en cada asignacion (asignados + [trabajo]). Copiar una lista de k
# elementos cuesta O(k), asi que una maquina que termina con n/m trabajos paga
# 1 + 2 + ... + n/m = O((n/m)^2) copias. Con m maquinas: O(n^2 / m).
# La version de la celda 2 guarda solo (carga, id) en el heap: O(n log m).


def list_scheduling_guia(trabajos, m):
    # Transcripcion literal del Listing 2 de la guia (solo para comparar).
    if m <= 0:
        raise ValueError("El numero de maquinas debe ser positivo")
    heap = [(0, i, []) for i in range(m)]
    heapq.heapify(heap)
    for trabajo in trabajos:
        carga, maquina, asignados = heapq.heappop(heap)
        nuevos = asignados + [trabajo]          # <- copia O(len(asignados))
        heapq.heappush(heap, (carga + trabajo, maquina, nuevos))
    resultado = sorted(heap, key=lambda x: x[1])
    planificacion = [asignados for _, _, asignados in resultado]
    return planificacion, max(carga for carga, _, _ in resultado)


# ---- 1. Equivalencia: ambas versiones producen EXACTAMENTE la misma salida --
generador = Random(SEMILLA_GLOBAL + 5)
iguales = 0
for _ in range(500):
    m = generador.randint(1, 8)
    t = [generador.randint(1, 50) for _ in range(generador.randint(0, 40))]
    if list_scheduling(t, m) == list_scheduling_guia(t, m):
        iguales += 1
print("Salidas identicas en %d de 500 instancias aleatorias." % iguales)
assert iguales == 500

# ---- 2. Tiempo al crecer n con pocas maquinas (m = 4) ----------------------
TAMANOS_LS = [1_000, 4_000, 16_000, 64_000]      # cada tamano es 4 veces el anterior
filas_ls = []
previo = None
for n in TAMANOS_LS:
    t = [Random(SEMILLA_GLOBAL + n).randint(1, 100) for _ in range(n)]
    _, t_opt = medir_mediana(list_scheduling, t, 4, repeticiones=3)
    _, t_guia = medir_mediana(list_scheduling_guia, t, 4, repeticiones=3)
    factor = "" if previo is None else "x%.1f / x%.1f" % (t_opt / previo[0], t_guia / previo[1])
    filas_ls.append([n, formato_tiempo(t_opt), formato_tiempo(t_guia),
                     "%.1f" % (t_guia / t_opt), factor])
    previo = (t_opt, t_guia)

mostrar_tabla(["n", "t optimizada", "t guia", "guia / optimizada",
               "crecimiento opt. / guia"], filas_ls,
              "List Scheduling con m = 4 (mediana de 3 ejecuciones)")
print("Al multiplicar n por 4, la version optimizada crece ~x4 (lineal) y la de la\n"
      "guia se acerca a ~x16 (cuadratica) cuando dominan las copias.")
guardar_csv("complejidad_list_scheduling.csv",
            ["n", "tiempo_optimizada", "tiempo_guia", "razon", "crecimiento"], filas_ls)

---
## 6. Generador reproducible de instancias

Cada instancia se genera con **su propia semilla** derivada de `SEMILLA_GLOBAL`, de modo que
puede regenerarse por separado. El conjunto completo se guarda además en
`resultados/instancias.json`.

| Familia | Distribución | Para qué sirve |
|---|---|---|
| `uniforme` | $p_j \sim U[1, 20]$ | duraciones variadas, caso típico |
| `estrecha` | $p_j \sim U[10, 20]$ | duraciones parecidas: difícil para LPT y para Ramificar-Podar |
| `sesgada` | 75 % en $U[1, 5]$, 25 % en $U[15, 30]$ | pocos trabajos largos: castiga a LS si llegan al final |
| `grande` | $p_j \sim U[1, 100]$ | instancias medianas y grandes del propuesto 2 |

**Propuesto 1:** $m \in \{2, 3, 4\}$, $n \in \{6, 9, 12, 15\}$, 3 familias, 2 instancias por
combinación (72) más las 6 instancias ajustadas → **78 instancias**.
**Propuesto 2:** $n \in \{10^2, 10^3, 10^4, 10^5\}$ con $m \in \{4, 16, 64\}$ y 30 permutaciones.

In [ ]:
# ============================================================================
# Celda 6 - Generador reproducible de instancias
# ============================================================================
# Cada instancia se genera con SU PROPIA semilla, derivada de SEMILLA_GLOBAL.
# Asi cualquier instancia puede regenerarse por separado conociendo solo su
# semilla, sin ejecutar las anteriores. Ademas el conjunto completo se guarda
# en JSON: aunque una version futura de Python cambiara el generador
# aleatorio, las instancias exactas del informe quedan conservadas.
#
# Familias (todas con enteros positivos):
#   * uniforme : p_j ~ U[1, 20]   duraciones muy variadas;
#   * estrecha : p_j ~ U[10, 20]  duraciones parecidas; es la familia dificil
#                para LPT y para Ramificar-Podar (muchos repartos casi iguales);
#   * sesgada  : 75 % cortos U[1, 5] y 25 % largos U[15, 30]; castiga a List
#                Scheduling cuando un trabajo largo llega al final.

FAMILIAS = ("uniforme", "estrecha", "sesgada")


def generar_trabajos(n, familia, semilla):
    # Devuelve n tiempos de proceso de la familia indicada.
    generador = Random(semilla)
    if familia == "uniforme":
        return [generador.randint(1, 20) for _ in range(n)]
    if familia == "estrecha":
        return [generador.randint(10, 20) for _ in range(n)]
    if familia == "sesgada":
        return [generador.randint(1, 5) if generador.random() < 0.75
                else generador.randint(15, 30) for _ in range(n)]
    if familia == "grande":                  # propuesto 2: U[1, 100]
        return [generador.randint(1, 100) for _ in range(n)]
    raise ValueError("Familia desconocida: " + familia)


# ---- Conjunto del propuesto 1: instancias pequenas (optimo calculable) -----
M_P1 = (2, 3, 4)
N_P1 = (6, 9, 12, 15)
INSTANCIAS_POR_COMBINACION = 2

instancias_p1 = []
contador = 0
for familia in FAMILIAS:
    for m in M_P1:
        for n in N_P1:
            for _ in range(INSTANCIAS_POR_COMBINACION):
                contador += 1
                semilla = SEMILLA_GLOBAL * 1000 + contador
                instancias_p1.append({
                    "id": "P1-%03d" % contador, "familia": familia, "n": n, "m": m,
                    "semilla": semilla, "trabajos": generar_trabajos(n, familia, semilla)})

# Instancias ajustadas (casos limite de la celda 4): alcanzan exactamente la
# garantia teorica de LS o de LPT. Sirven para comprobar que la razon
# observada puede TOCAR la garantia, pero nunca superarla.
for m in M_P1:
    for nombre, funcion in (("ajustada_ls", instancia_ajustada_ls),
                            ("ajustada_lpt", instancia_ajustada_lpt)):
        contador += 1
        t = funcion(m)
        instancias_p1.append({"id": "P1-%03d" % contador, "familia": nombre, "n": len(t),
                              "m": m, "semilla": None, "trabajos": t})

# ---- Conjunto del propuesto 2: instancias medianas y grandes ---------------
# Un mismo conjunto de trabajos por tamano n se evalua con varios m.
N_P2 = (100, 1_000, 10_000, 100_000)
M_P2 = (4, 16, 64)
PERMUTACIONES = 30                           # la guia exige al menos 20

conjuntos_p2 = []
for n in N_P2:
    semilla = SEMILLA_GLOBAL * 1000 + 500 + len(conjuntos_p2)
    conjuntos_p2.append({"id": "P2-n%d" % n, "familia": "grande", "n": n,
                         "semilla": semilla, "trabajos": generar_trabajos(n, "grande", semilla)})

# ---- Conservacion del conjunto ---------------------------------------------
ruta_json = os.path.join(CARPETA_RESULTADOS, "instancias.json")
with open(ruta_json, "w", encoding="utf-8") as archivo:
    json.dump({"semilla_global": SEMILLA_GLOBAL,
               "propuesto_1": instancias_p1,
               # Para el propuesto 2 se guardan semilla y parametros; los
               # trabajos se regeneran con generar_trabajos(n, "grande", semilla).
               "propuesto_2": [{k: v for k, v in c.items() if k != "trabajos"}
                               for c in conjuntos_p2]}, archivo)
print("Propuesto 1: %d instancias (%d aleatorias + %d ajustadas)"
      % (len(instancias_p1), contador - 6, 6))
print("Propuesto 2: %d conjuntos de trabajos x %d valores de m" % (len(conjuntos_p2), len(M_P2)))
print("Guardado: " + ruta_json)

# Comprobacion de reproducibilidad: regenerar una instancia con su semilla
# debe dar exactamente los mismos trabajos.
muestra = instancias_p1[17]
assert generar_trabajos(muestra["n"], muestra["familia"], muestra["semilla"]) == muestra["trabajos"]
print("\nEjemplo %s (%s, n=%d, m=%d, semilla=%d): %s" % (
    muestra["id"], muestra["familia"], muestra["n"], muestra["m"], muestra["semilla"],
    muestra["trabajos"]))

---
## 7. Propuesto 1 — evaluación de calidad frente al óptimo

Para cada instancia se ejecutan List Scheduling (orden generado), LPT y Ramificar-Podar, y se calcula
$$r(I) = \frac{\mathrm{ALG}(I)}{\mathrm{OPT}(I)}, \qquad
\delta(I) = \frac{\mathrm{ALG}(I) - \mathrm{OPT}(I)}{\mathrm{OPT}(I)} \times 100\,\%.$$

Las garantías se verifican con **aritmética entera exacta** ($m \cdot C_{LS} \le (2m-1)\,\mathrm{OPT}$
y $3m \cdot C_{LPT} \le (4m-1)\,\mathrm{OPT}$), para que un redondeo no oculte ni invente una violación.

In [ ]:
# ============================================================================
# Celda 7 - Propuesto 1: evaluacion de calidad frente al optimo
# ============================================================================
# Para cada instancia pequena: List Scheduling (orden generado), LPT y
# Ramificar-Podar. Se registran LB, OPT, C_LS, C_LPT, razones, desviaciones,
# tiempos y nodos del metodo exacto.
#
#   r(I)     = ALG(I) / OPT(I)                 razon observada (>= 1)
#   delta(I) = (ALG(I) - OPT(I)) / OPT(I) %    desviacion relativa
#   garantia LS  = 2 - 1/m
#   garantia LPT = 4/3 - 1/(3m)
# Las razones se comparan con aritmetica EXACTA de enteros (ALG * 3m <= ...),
# para que un error de redondeo no oculte ni invente una violacion.

LIMITE_NODOS_P1 = 5_000_000                  # tope de seguridad del exacto


def garantia_ls(m):
    return 2 - 1 / m


def garantia_lpt(m):
    return 4 / 3 - 1 / (3 * m)


def respeta_garantia_ls(c, opt, m):
    # C_LS <= (2 - 1/m) OPT   <=>   m * C_LS <= (2m - 1) * OPT
    return m * c <= (2 * m - 1) * opt


def respeta_garantia_lpt(c, opt, m):
    # C_LPT <= (4/3 - 1/(3m)) OPT   <=>   3m * C_LPT <= (4m - 1) * OPT
    return 3 * m * c <= (4 * m - 1) * opt


filas_p1 = []
for inst in instancias_p1:
    t, m = inst["trabajos"], inst["m"]
    validar_instancia(t, m)

    # --- Ejecucion y medicion (mediana de REPETICIONES) ---------------------
    (plan_ls, c_ls), t_ls = medir_mediana(list_scheduling, t, m)
    (plan_lpt, c_lpt), t_lpt = medir_mediana(lpt, t, m)
    (plan_bb, opt, nodos), t_bb = medir_mediana(branch_and_bound, t, m, LIMITE_NODOS_P1)
    lb = cota_inferior(t, m)

    # --- Requisito 1: validar TODAS las planificaciones ---------------------
    assert verificar_solucion(t, m, plan_ls, c_ls), inst["id"]
    assert verificar_solucion(t, m, plan_lpt, c_lpt), inst["id"]
    assert verificar_solucion(t, m, plan_bb, opt), inst["id"]
    assert lb <= opt <= min(c_ls, c_lpt), inst["id"]

    filas_p1.append({
        "id": inst["id"], "familia": inst["familia"], "n": inst["n"], "m": m,
        "LB": lb, "OPT": opt, "C_LS": c_ls, "C_LPT": c_lpt,
        "r_LS": c_ls / opt, "r_LPT": c_lpt / opt,
        "dev_LS": 100 * (c_ls - opt) / opt, "dev_LPT": 100 * (c_lpt - opt) / opt,
        "t_LS": t_ls, "t_LPT": t_lpt, "t_BB": t_bb, "nodos": nodos,
        "LS_optimo": c_ls == opt, "LPT_optimo": c_lpt == opt, "OPT_igual_LB": opt == lb,
        "garantia_LS_ok": respeta_garantia_ls(c_ls, opt, m),
        "garantia_LPT_ok": respeta_garantia_lpt(c_lpt, opt, m),
    })

columnas_p1 = list(filas_p1[0].keys())
guardar_csv("propuesto1_calidad.csv", columnas_p1,
            [[("%.9g" % f[c]) if isinstance(f[c], float) else f[c] for c in columnas_p1]
             for f in filas_p1])

mostrar_tabla(
    ["id", "familia", "n", "m", "LB", "OPT", "C_LS", "C_LPT", "r_LS", "r_LPT",
     "nodos", "t BB"],
    [[f["id"], f["familia"], f["n"], f["m"], f["LB"], f["OPT"], f["C_LS"], f["C_LPT"],
      "%.3f" % f["r_LS"], "%.3f" % f["r_LPT"], f["nodos"], formato_tiempo(f["t_BB"])]
     for f in filas_p1],
    "Propuesto 1: %d instancias, todas las planificaciones validadas" % len(filas_p1),
    max_filas=20)

### Resumen, garantías y pregunta de análisis

**¿LPT domina a List Scheduling en todas las instancias?** No. LPT es mejor en la gran
mayoría, pero hay instancias donde el orden generado le da a LS un makespan **menor** que LPT.

La explicación: para cualquier instancia existe un orden con el que LS construye un óptimo
(basta listar los trabajos máquina por máquina según una planificación óptima). LPT fija
**un** orden particular; cuando ese orden no es el adecuado, algún otro orden puede ganarle.
La celda lo muestra recorriendo **todos** los órdenes de las instancias ajustadas de LPT.

In [ ]:
# ============================================================================
# Celda 8 - Propuesto 1: resumen, garantias y dominancia
# ============================================================================
aleatorias = [f for f in filas_p1 if not f["familia"].startswith("ajustada")]
ajustadas = [f for f in filas_p1 if f["familia"].startswith("ajustada")]


def resumir(grupo, etiqueta):
    # Una fila por algoritmo: veces optimo, razon promedio/mediana/maxima.
    filas = []
    for alg, gar in (("LS", garantia_ls), ("LPT", garantia_lpt)):
        razones = [f["r_" + alg] for f in grupo]
        desv = [f["dev_" + alg] for f in grupo]
        # Garantia del grupo: si mezcla varios m, se informa el rango.
        gs = sorted({gar(f["m"]) for f in grupo})
        texto_gar = "%.3f" % gs[0] if len(gs) == 1 else "%.3f - %.3f" % (gs[0], gs[-1])
        filas.append([etiqueta, alg, len(grupo),
                      "%d (%.0f %%)" % (sum(f[alg + "_optimo"] for f in grupo),
                                        100 * sum(f[alg + "_optimo"] for f in grupo) / len(grupo)),
                      "%.4f" % mean(razones), "%.4f" % median(razones), "%.4f" % max(razones),
                      "%.2f" % mean(desv), texto_gar,
                      sum(not f["garantia_%s_ok" % alg] for f in grupo)])
    return filas


encabezado_resumen = ["grupo", "alg", "instancias", "veces optimo", "r promedio",
                      "r mediana", "r maxima", "desv. prom. %", "garantia", "violaciones"]
resumen_p1 = []
for m in M_P1:
    resumen_p1 += resumir([f for f in aleatorias if f["m"] == m], "aleatorias m=%d" % m)
resumen_p1 += resumir(aleatorias, "aleatorias (todas)")
resumen_p1 += resumir(ajustadas, "ajustadas")
mostrar_tabla(encabezado_resumen, resumen_p1, "Resumen de calidad observada")
guardar_csv("propuesto1_resumen.csv", encabezado_resumen, resumen_p1)

# ---- Requisito 7: ninguna razon observada viola la garantia ----------------
violaciones = sum(not f["garantia_LS_ok"] or not f["garantia_LPT_ok"] for f in filas_p1)
print("\nViolaciones de garantia en %d instancias: %d" % (len(filas_p1), violaciones))
assert violaciones == 0
# Las ajustadas deben alcanzar la garantia EXACTAMENTE (igualdad):
for f in ajustadas:
    alg = "LS" if f["familia"] == "ajustada_ls" else "LPT"
    gar = garantia_ls(f["m"]) if alg == "LS" else garantia_lpt(f["m"])
    print("  %s m=%d: r_%s = %d/%d = %.4f  | garantia = %.4f" % (
        f["familia"], f["m"], alg, f["C_" + alg], f["OPT"], f["r_" + alg], gar))

# ---- Cota inferior como certificado ----------------------------------------
certificadas = sum(f["OPT_igual_LB"] for f in filas_p1)
sin_explorar = sum(f["nodos"] == 0 for f in filas_p1)
print("\nOPT = LB en %d de %d instancias; en %d el exacto no exploro ningun nodo\n"
      "(LPT ya alcanzaba la cota y quedo certificado como optimo)."
      % (certificadas, len(filas_p1), sin_explorar))

# ---- Pregunta de analisis: LPT domina a LS? --------------------------------
lpt_mejor = [f for f in aleatorias if f["C_LPT"] < f["C_LS"]]
empates = [f for f in aleatorias if f["C_LPT"] == f["C_LS"]]
ls_mejor = [f for f in aleatorias if f["C_LS"] < f["C_LPT"]]
print("\nComparacion directa en %d instancias aleatorias:" % len(aleatorias))
print("  LPT mejor que LS : %d" % len(lpt_mejor))
print("  empate           : %d (de ellos, ambos optimos: %d)"
      % (len(empates), sum(f["LS_optimo"] for f in empates)))
print("  LS mejor que LPT : %d  <- excepciones" % len(ls_mejor))
mostrar_tabla(["id", "familia", "n", "m", "trabajos (orden generado)", "OPT", "C_LS", "C_LPT"],
              [[f["id"], f["familia"], f["n"], f["m"],
                str(next(i["trabajos"] for i in instancias_p1 if i["id"] == f["id"])),
                f["OPT"], f["C_LS"], f["C_LPT"]] for f in ls_mejor],
              "Instancias donde List Scheduling supera a LPT")

# ---- Por que ocurre: TODOS los ordenes de la instancia ajustada de LPT ------
# Existe siempre un orden para el cual LS construye un optimo (basta listar
# los trabajos maquina por maquina segun una planificacion optima). LPT fija
# UN orden; si ese orden no es bueno para la instancia, algun orden
# particular de LS puede ganarle.
analisis_ordenes = []
for m in (2, 3):
    t = instancia_ajustada_lpt(m)
    _, c_lpt = lpt(t, m)
    _, opt, _ = branch_and_bound(t, m)
    conteo = Counter(list_scheduling(list(orden), m)[1] for orden in permutations(t))
    total = sum(conteo.values())
    mejores = sum(v for k, v in conteo.items() if k < c_lpt)
    analisis_ordenes.append([m, str(t), total, opt, c_lpt,
                             str(dict(sorted(conteo.items()))),
                             "%d (%.1f %%)" % (mejores, 100 * mejores / total)])
mostrar_tabla(["m", "trabajos", "ordenes", "OPT", "C_LPT", "makespan LS -> n.o de ordenes",
               "ordenes donde LS < LPT"], analisis_ordenes,
              "\nInstancia ajustada de LPT: List Scheduling con TODOS los ordenes posibles")
guardar_csv("propuesto1_ordenes_ajustada_lpt.csv",
            ["m", "trabajos", "ordenes", "OPT", "C_LPT", "distribucion", "LS_mejor_que_LPT"],
            analisis_ordenes)

**Lectura.** Ninguna razón observada supera su garantía, y las instancias ajustadas la
**alcanzan con igualdad**. Esto *es compatible* con los teoremas pero **no los demuestra**:
un experimento solo puede refutar una garantía (encontrando un contraejemplo), nunca
probarla para todas las instancias. La demostración es la de Graham; el experimento
comprueba que la implementación se comporta como predice la teoría.

### Gráfico de calidad observada

Cada punto es una instancia aleatoria; la raya negra es la mediana; el segmento punteado,
la garantía; los marcadores huecos, las instancias ajustadas.

In [ ]:
# ============================================================================
# Celda 9 - Propuesto 1: grafico de calidad observada frente a la garantia
# ============================================================================
# Cada punto es una instancia aleatoria (con un leve desplazamiento horizontal
# para que no se oculten entre si). La raya horizontal gruesa es la mediana.
# El segmento punteado marca la garantia teorica de cada algoritmo para ese m.
# Los marcadores huecos son las instancias ajustadas: tocan la garantia.

figura, eje = plt.subplots(figsize=(8.6, 5.2))
jitter = Random(SEMILLA_GLOBAL)
desplazamiento = {"LS": -0.18, "LPT": 0.18}

for alg in ("LS", "LPT"):
    for m in M_P1:
        x0 = m + desplazamiento[alg]
        grupo = [f["r_" + alg] for f in aleatorias if f["m"] == m]
        xs = [x0 + jitter.uniform(-0.06, 0.06) for _ in grupo]
        eje.scatter(xs, grupo, s=26, color=COLOR[alg], marker=MARCADOR[alg], alpha=0.75,
                    edgecolors="white", linewidths=0.8, zorder=3,
                    label=("List Scheduling" if alg == "LS" else "LPT") if m == M_P1[0] else None)
        eje.plot([x0 - 0.1, x0 + 0.1], [median(grupo)] * 2, color=TINTA, linewidth=2, zorder=4)

        # Garantia teorica (umbral, por eso va punteada) con etiqueta directa.
        gar = garantia_ls(m) if alg == "LS" else garantia_lpt(m)
        eje.plot([x0 - 0.13, x0 + 0.13], [gar, gar], color=COLOR[alg], linestyle=(0, (3, 2)),
                 linewidth=1.5, zorder=2)
        eje.annotate("%.3f" % gar, (x0, gar), xytext=(0, 8), textcoords="offset points",
                     ha="center", va="bottom", fontsize=8, color=TINTA_SECUNDARIA)

        # Instancia ajustada correspondiente (hueca).
        for f in ajustadas:
            if f["m"] == m and f["familia"] == ("ajustada_ls" if alg == "LS" else "ajustada_lpt"):
                eje.scatter([x0], [f["r_" + alg]], s=70, facecolors="white",
                            edgecolors=COLOR[alg], marker=MARCADOR[alg], linewidths=1.8, zorder=5,
                            label="Instancia ajustada" if (m == M_P1[0] and alg == "LS") else None)

eje.plot([], [], color=TINTA, linewidth=2, label="Mediana")
eje.plot([], [], color=TINTA_TENUE, linestyle=(0, (3, 2)), linewidth=1.5, label="Garantía teórica")
eje.set_xticks(list(M_P1))
eje.set_xticklabels(["m = %d" % m for m in M_P1])
eje.set_xlim(1.5, 4.5)
eje.set_ylim(0.97, 1.86)
eje.grid(axis="x", visible=False)
eje.set_ylabel("razón observada  r(I) = ALG / OPT")
eje.set_title("Calidad observada frente a la garantía teórica (%d instancias aleatorias)"
              % len(aleatorias), loc="left")
eje.legend(loc="upper left", ncol=3)
guardar_figura(figura, "p1_calidad_razones.png")

### Rendimiento y límite práctico del método exacto

Se usa la familia `estrecha` con $m = 4$ y $n = 10, 11, \dots, 24$ (6 instancias por tamaño),
con un tope de un millón de nodos por ejecución.

In [ ]:
# ============================================================================
# Celda 10 - Propuesto 1: rendimiento y limite practico del metodo exacto
# ============================================================================
# Familia "estrecha" (duraciones parecidas) con m = 4: el caso mas dificil del
# rango del propuesto 1. Hay muchisimos repartos casi iguales y la poda actua
# tarde. Cada ejecucion tiene un tope de nodos para no bloquear la sesion; si
# se alcanza, la instancia se registra como "abortada" (el optimo no se
# certifico dentro del presupuesto).

LIMITE_NODOS_ESCALA = 1_000_000
M_ESCALA = 4
N_ESCALA = list(range(10, 25))                # 10, 11, ..., 24
INSTANCIAS_ESCALA = 6

filas_escala = []
for n in N_ESCALA:
    for rep in range(INSTANCIAS_ESCALA):
        semilla = SEMILLA_GLOBAL * 1000 + 900 + 10 * n + rep
        t = generar_trabajos(n, "estrecha", semilla)
        inicio = perf_counter()
        try:
            plan, opt, nodos = branch_and_bound(t, M_ESCALA, LIMITE_NODOS_ESCALA)
            estado = "ok"
            assert verificar_solucion(t, M_ESCALA, plan, opt)
        except LimiteNodosExcedido as error:
            opt, nodos, estado = None, error.nodos, "abortada"
        duracion = perf_counter() - inicio
        filas_escala.append([M_ESCALA, n, rep, semilla, cota_inferior(t, M_ESCALA), opt,
                             lpt(t, M_ESCALA)[1], nodos, "%.9g" % duracion, estado])

guardar_csv("propuesto1_escala_exacto.csv",
            ["m", "n", "rep", "semilla", "LB", "OPT", "C_LPT", "nodos", "tiempo_s", "estado"],
            filas_escala)

resumen_escala = []
for n in N_ESCALA:
    g = [f for f in filas_escala if f[1] == n]
    nodos = sorted(f[7] for f in g)
    # Una ejecucion abortada solo dice "mas de LIMITE nodos". Si alguna cae en
    # el centro de la muestra, la mediana verdadera es mayor: se marca con ">=".
    centro = nodos[(len(nodos) - 1) // 2: len(nodos) // 2 + 1]
    prefijo = ">= " if any(v > LIMITE_NODOS_ESCALA for v in centro) else ""
    maximo = nodos[-1]
    resumen_escala.append([n, sum(v == 0 for v in nodos), prefijo + "%g" % median(nodos),
                           "tope" if maximo > LIMITE_NODOS_ESCALA else maximo,
                           formato_tiempo(max(float(f[8]) for f in g)),
                           sum(f[9] == "abortada" for f in g)])
mostrar_tabla(["n", "certificadas en la raiz", "nodos mediana", "nodos max", "t max",
               "abortadas"], resumen_escala,
              "Ramificar-Podar, m = %d, familia estrecha (%d instancias por n)"
              % (M_ESCALA, INSTANCIAS_ESCALA))

# Costo por nodo: si el tiempo es proporcional a los nodos, "nodos" es una
# medida de trabajo fiel e independiente del equipo.
con_trabajo = [f for f in filas_escala if f[7] >= 1000 and f[9] == "ok"]
us_por_nodo = median(1e6 * float(f[8]) / f[7] for f in con_trabajo)
print("\nCosto mediano por nodo (instancias con >= 1000 nodos): %.2f us" % us_por_nodo)

# ---- Grafico de rendimiento -------------------------------------------------
figura, (eje_n, eje_t) = plt.subplots(1, 2, figsize=(10.5, 4.3),
                                      gridspec_kw={"width_ratios": [1.45, 1]})
jitter = Random(SEMILLA_GLOBAL + 10)
resueltas = [f for f in filas_escala if f[9] == "ok"]
abortadas = [f for f in filas_escala if f[9] == "abortada"]

# Panel 1: nodos (+1 para poder dibujar el 0 en escala logaritmica).
eje_n.scatter([f[1] + jitter.uniform(-0.18, 0.18) for f in resueltas],
              [f[7] + 1 for f in resueltas], s=22, color=COLOR["BB"], marker=MARCADOR["BB"],
              alpha=0.7, edgecolors="white", linewidths=0.6, zorder=3, label="cada instancia")
if abortadas:
    eje_n.scatter([f[1] for f in abortadas], [LIMITE_NODOS_ESCALA + 1] * len(abortadas),
                  s=40, color=TINTA, marker="x", zorder=4, label="abortada por el tope")
eje_n.plot(N_ESCALA, [max(f[7] for f in filas_escala if f[1] == n) + 1 for n in N_ESCALA],
           color=TINTA, linewidth=1.6, zorder=2, label="peor instancia de cada n")
eje_n.axhline(LIMITE_NODOS_ESCALA, color=TINTA_TENUE, linestyle=(0, (3, 2)), linewidth=1.2)
eje_n.annotate("tope de seguridad: 1 000 000 de nodos", (N_ESCALA[0], LIMITE_NODOS_ESCALA),
               xytext=(0, 5), textcoords="offset points", fontsize=8, color=TINTA_SECUNDARIA)
eje_n.set_yscale("log")
eje_n.set_ylim(0.6, 1e7)
eje_n.set_xticks(N_ESCALA[::2])
eje_n.set_xlabel("n (número de trabajos)")
eje_n.set_ylabel("nodos explorados + 1 (escala log)")
eje_n.set_title("Nodos explorados, m = %d, familia estrecha" % M_ESCALA, loc="left")
eje_n.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3)
eje_n.annotate("0 nodos: LPT alcanza LB y queda\ncertificado en la raíz (n = 12, 16, 20, 24)",
               (12.25, 1), xytext=(12.75, 1.9), textcoords="data", ha="left", va="bottom",
               fontsize=8, color=TINTA_SECUNDARIA)

# Panel 2: tiempo frente a nodos (log-log). Pendiente ~1 => tiempo proporcional.
xs = [f[7] for f in resueltas if f[7] > 0]
ys = [float(f[8]) for f in resueltas if f[7] > 0]
eje_t.scatter(xs, ys, s=22, color=COLOR["BB"], marker=MARCADOR["BB"], alpha=0.7,
              edgecolors="white", linewidths=0.6, zorder=3)
referencia = [min(xs), max(xs)]
eje_t.plot(referencia, [x * us_por_nodo * 1e-6 for x in referencia], color=TINTA,
           linewidth=1.2, zorder=2, label="%.1f µs por nodo" % us_por_nodo)
eje_t.set_xscale("log")
eje_t.set_yscale("log")
eje_t.set_xlabel("nodos explorados (escala log)")
eje_t.set_ylabel("tiempo (s, escala log)")
eje_t.set_title("Tiempo frente a nodos", loc="left")
eje_t.legend(loc="upper left")
figura.tight_layout()
guardar_figura(figura, "p1_rendimiento_exacto.png")

**Lectura.** El costo del método exacto depende **más de la instancia que de $n$**:

- Cuando $n$ es múltiplo de $m$ (12, 16, 20, 24), cada máquina recibe la misma cantidad de
  trabajos parecidos, LPT alcanza $\lceil \sum p / m \rceil$ y el óptimo queda **certificado en
  la raíz** con 0 nodos.
- En los demás $n$, alguna máquina debe llevar un trabajo más y LPT **no** fue óptimo en
  ninguna de las instancias. El método tiene que **buscar** un reparto mejor entre
  muchísimos repartos casi iguales. Cuando lo encuentra y ese reparto iguala $LB$, la búsqueda
  termina (todo lo demás queda podado); cuando $\mathrm{OPT} > LB$ (casi todas las de
  $n = 10$ y $11$), además debe **demostrar** que nada alcanza la cota. La peor instancia
  crece de forma exponencial hasta tocar el tope en $n = 22$ y $23$.
- El tiempo es proporcional a los nodos (pendiente 1 en escala log-log), así que "nodos" es
  una métrica de trabajo fiel e independiente del equipo.

---
## 8. Propuesto 2 — efecto del orden y escalabilidad

Instancias medianas y grandes donde **no** se calcula el óptimo. Para cada conjunto se compara
el orden original, 30 permutaciones aleatorias con LS y el orden LPT.

**¿Por qué $C_{ALG}/LB$ no equivale a $C_{ALG}/\mathrm{OPT}$?** Como $LB \le \mathrm{OPT}$,
$$\frac{C_{ALG}}{\mathrm{OPT}} \le \frac{C_{ALG}}{LB}:$$
el cociente contra la cota es solo una **cota superior** de la razón real. Si $LB < \mathrm{OPT}$
(por ejemplo, tres trabajos de 2 en dos máquinas: $LB = 3$, $\mathrm{OPT} = 4$), un algoritmo
óptimo muestra $C/LB = 4/3 > 1$. En cambio, cuando $C_{ALG} = LB$ la conclusión es fuerte:
el algoritmo es **óptimo** en esa instancia.

In [ ]:
# ============================================================================
# Celda 11 - Propuesto 2: efecto del orden y escalabilidad
# ============================================================================
# Para cada conjunto de trabajos (n = 100 ... 100 000) y cada m (4, 16, 64):
#   * List Scheduling con el orden original (el del generador);
#   * List Scheduling con PERMUTACIONES ordenes aleatorios (semilla fija);
#   * LPT (ordenar de mayor a menor + List Scheduling).
# Aqui el optimo NO se calcula (seria inviable): la calidad se mide contra la
# cota inferior, C_ALG / LB, que es una COTA SUPERIOR de la razon real
# C_ALG / OPT porque LB <= OPT.
#
# Tiempos: mediana de REPETICIONES_P2 ejecuciones. Se mide por separado el
# ordenamiento para saber cuanto de LPT es "ordenar" y cuanto es "asignar".

REPETICIONES_P2 = 5

filas_p2 = []            # una fila por (n, m) con el resumen
makespans_perm = {}      # (n, m) -> lista de makespans de las permutaciones

for conjunto in conjuntos_p2:
    t = conjunto["trabajos"]
    n = conjunto["n"]

    # Permutaciones reproducibles: se generan UNA vez por conjunto y se reusan
    # para todos los m, de modo que cada m ve exactamente los mismos ordenes.
    generador = Random(conjunto["semilla"] + 1)
    ordenes = []
    for _ in range(PERMUTACIONES):
        orden = list(t)
        generador.shuffle(orden)
        ordenes.append(orden)

    # Costo del ordenamiento (independiente de m).
    _, t_orden = medir_mediana(sorted, t, repeticiones=REPETICIONES_P2)

    for m in M_P2:
        lb = cota_inferior(t, m)
        (plan_ls, c_ls), t_ls = medir_mediana(list_scheduling, t, m, repeticiones=REPETICIONES_P2)
        (plan_lpt, c_lpt), t_lpt = medir_mediana(lpt, t, m, repeticiones=REPETICIONES_P2)
        assert verificar_solucion(t, m, plan_ls, c_ls)
        assert verificar_solucion(t, m, plan_lpt, c_lpt)

        valores = []
        for orden in ordenes:
            plan, c = list_scheduling(orden, m)
            assert c >= lb and makespan(plan) == c    # validacion ligera
            valores.append(c)
        makespans_perm[(n, m)] = valores

        filas_p2.append({
            "n": n, "m": m, "LB": lb,
            "C_original": c_ls, "C_LPT": c_lpt,
            "C_perm_min": min(valores), "C_perm_mediana": median(valores),
            "C_perm_max": max(valores),
            "q_original": c_ls / lb, "q_LPT": c_lpt / lb,
            "q_perm_min": min(valores) / lb, "q_perm_mediana": median(valores) / lb,
            "q_perm_max": max(valores) / lb,
            "rango_perm_pct": 100 * (max(valores) - min(valores)) / lb,
            "t_LS": t_ls, "t_LPT": t_lpt, "t_orden": t_orden,
            "sobrecosto_LPT_pct": 100 * (t_lpt - t_ls) / t_ls,
            "mejora_LPT_pct": 100 * (median(valores) - c_lpt) / median(valores),
        })

columnas_p2 = list(filas_p2[0].keys())
guardar_csv("propuesto2_escalabilidad.csv", columnas_p2,
            [[("%.9g" % f[c]) if isinstance(f[c], float) else f[c] for c in columnas_p2]
             for f in filas_p2])
guardar_csv("propuesto2_makespans_permutaciones.csv", ["n", "m", "permutacion", "makespan"],
            [[n, m, k + 1, v] for (n, m), vals in makespans_perm.items()
             for k, v in enumerate(vals)])

mostrar_tabla(
    ["n", "m", "LB", "C original", "C perm (min-med-max)", "C LPT",
     "C/LB original", "C/LB perm. med.", "C/LB LPT", "t LS", "t LPT", "t ordenar"],
    [[f["n"], f["m"], f["LB"], f["C_original"],
      "%d - %g - %d" % (f["C_perm_min"], f["C_perm_mediana"], f["C_perm_max"]), f["C_LPT"],
      "%.4f" % f["q_original"], "%.4f" % f["q_perm_mediana"], "%.4f" % f["q_LPT"],
      formato_tiempo(f["t_LS"]), formato_tiempo(f["t_LPT"]), formato_tiempo(f["t_orden"])]
     for f in filas_p2],
    "Propuesto 2: %d permutaciones aleatorias por conjunto" % PERMUTACIONES)

### Instancia donde el orden afecta visiblemente a List Scheduling

Para **cualquier** orden, LS cumple (Graham, 1966)
$$C_{LS} \le \frac{\sum_j p_j}{m} + \left(1 - \frac{1}{m}\right) p_{\max} \le LB + p_{\max},$$
porque el trabajo que termina último empezó cuando su máquina era la menos cargada (a lo sumo
el promedio). Por eso el efecto del orden está acotado por $p_{\max}/LB$: es grande con pocos
trabajos por máquina y desaparece cuando $n/m$ crece.

In [ ]:
# ============================================================================
# Celda 12 - Propuesto 2: instancia donde el orden afecta visiblemente a LS
# ============================================================================
# Por que el efecto del orden depende de n/m: para CUALQUIER orden, List
# Scheduling cumple (Graham, 1966)
#       C_LS <= sum(p)/m + (1 - 1/m) * p_max  <=  LB + p_max,
# porque el trabajo que termina ultimo empezo cuando su maquina era la menos
# cargada (a lo sumo el promedio). Entonces
#       C_LS / LB <= 1 + p_max / LB   para todo orden.
# Con pocos trabajos por maquina, p_max es comparable con LB y el orden pesa
# mucho; con muchos trabajos por maquina, p_max / LB -> 0 y todos los ordenes
# dan casi lo mismo.

PERMUTACIONES_ENFOCADAS = 1000
N_ENFOCADO = N_P2[0]                                  # n = 100
t_enfocado = conjuntos_p2[0]["trabajos"]
generador = Random(SEMILLA_GLOBAL + 12)

enfoque = {}
filas_enfoque = []
for m in M_P2:
    lb = cota_inferior(t_enfocado, m)
    valores = []
    for _ in range(PERMUTACIONES_ENFOCADAS):
        orden = list(t_enfocado)
        generador.shuffle(orden)
        valores.append(list_scheduling(orden, m)[1])
    c_original = list_scheduling(t_enfocado, m)[1]
    c_lpt = lpt(t_enfocado, m)[1]
    c_asc = list_scheduling(sorted(t_enfocado), m)[1]     # orden ascendente
    cota_graham = lb + max(t_enfocado)
    enfoque[m] = {"lb": lb, "valores": valores, "original": c_original, "lpt": c_lpt,
                  "ascendente": c_asc}
    assert max(valores) <= cota_graham                    # la cota se cumple
    filas_enfoque.append([m, lb, c_lpt, c_original, c_asc, min(valores),
                          median(valores), max(valores),
                          "%.1f" % (100 * (max(valores) - min(valores)) / lb),
                          "%d (%.1f)" % (cota_graham, cota_graham / lb),
                          "%.1f %%" % (100 * sum(v > c_lpt for v in valores) / len(valores))])

mostrar_tabla(["m", "LB", "C_LPT", "C original", "C ascendente", "perm min", "perm mediana",
               "perm max", "rango % de LB", "cota LB + p_max (razon)", "ordenes peores que LPT"],
              filas_enfoque,
              "n = %d trabajos, %d ordenes aleatorios de List Scheduling"
              % (N_ENFOCADO, PERMUTACIONES_ENFOCADAS))
guardar_csv("propuesto2_orden_visible.csv",
            ["m", "LB", "C_LPT", "C_original", "C_ascendente", "perm_min", "perm_mediana",
             "perm_max", "rango_pct_LB", "cota_LB_mas_pmax", "pct_ordenes_peores_que_LPT"],
            filas_enfoque)
for m in M_P2:
    if enfoque[m]["lpt"] == enfoque[m]["lb"]:
        print("m = %d: LPT alcanza LB -> es OPTIMO (certificado por la cota)." % m)
    else:
        print("m = %d: LPT = %d > LB = %d -> el optimo esta entre %d y %d (no certificado)."
              % (m, enfoque[m]["lpt"], enfoque[m]["lb"], enfoque[m]["lb"], enfoque[m]["lpt"]))

### Gráficos del propuesto 2

In [ ]:
# ============================================================================
# Celda 13 - Propuesto 2: graficos de distribucion, escala y tiempo
# ============================================================================
# Rampa ordinal azul para m (todas son ejecuciones de List Scheduling, que es
# azul en todo el notebook): mas maquinas -> azul mas oscuro.
AZUL_M = {4: "#86b6ef", 16: "#2a78d6", 64: "#104281"}
MARCA_M = {4: "o", 16: "s", 64: "D"}

# ---- Grafico 1: distribucion del makespan segun el orden (n = 100) ---------
figura, ejes = plt.subplots(1, 3, figsize=(11.5, 4.1))
for eje, m in zip(ejes, M_P2):
    e = enfoque[m]
    valores = e["valores"]
    bordes = range(min(valores), max(valores) + 2)
    eje.hist(valores, bins=bordes, color=COLOR["LS"], alpha=0.85, edgecolor="white",
             linewidth=0.6, label="LS, %d órdenes aleatorios" % len(valores))
    eje.axvline(e["lb"], color=TINTA_TENUE, linestyle=(0, (3, 2)), linewidth=1.3,
                label="cota inferior LB")
    eje.axvline(e["lpt"], color=COLOR["LPT"], linewidth=2.2, label="LPT")
    eje.axvline(e["original"], color=TINTA, linewidth=1.3, linestyle=(0, (1, 1.5)),
                label="LS, orden original")
    # Espacio libre sobre las barras para las etiquetas directas (texto en
    # tinta con fondo blanco, nunca en el color de la serie).
    altura = max(Counter(valores).values())
    eje.set_ylim(0, altura * 1.32)
    fondo = dict(boxstyle="square,pad=0.15", facecolor="white", edgecolor="none")
    texto_lpt = "LPT = LB = %d" % e["lpt"] if e["lpt"] == e["lb"] else "LPT = %d" % e["lpt"]
    eje.annotate(texto_lpt, (e["lpt"], altura * 1.27), xytext=(4, 0), textcoords="offset points",
                 ha="left", va="center", fontsize=8, color=TINTA_SECUNDARIA, bbox=fondo)
    eje.annotate("original = %d" % e["original"], (e["original"], altura * 1.12),
                 xytext=(4, 0), textcoords="offset points", ha="left", va="center",
                 fontsize=8, color=TINTA_SECUNDARIA, bbox=fondo)
    eje.set_title("m = %d   (n/m = %.1f)" % (m, N_ENFOCADO / m), loc="left")
    eje.set_xlabel("makespan")
    eje.grid(axis="x", visible=False)
    # Margen a la izquierda para que LB y LPT no queden pegados al borde.
    ancho = max(valores) - e["lb"]
    eje.set_xlim(e["lb"] - 0.06 * ancho, max(valores) + 0.04 * ancho)
ejes[0].set_ylabel("número de órdenes")
manejadores, etiquetas = ejes[1].get_legend_handles_labels()
figura.legend(manejadores, etiquetas, loc="lower center", ncol=4)
figura.suptitle("n = %d trabajos: el mismo conjunto, distinto orden, distinto makespan"
                % N_ENFOCADO, x=0.01, ha="left", fontsize=10, color=TINTA_SECUNDARIA)
figura.tight_layout(rect=(0, 0.07, 1, 1))
guardar_figura(figura, "p2_distribucion_orden.png")

# ---- Grafico 2: el efecto del orden se desvanece al crecer n/m -------------
# Exceso sobre la cota inferior: 100 * (C - LB) / LB, en escala logaritmica.
figura, eje = plt.subplots(figsize=(8.6, 4.8))
for m in M_P2:
    g = [f for f in filas_p2 if f["m"] == m]
    ns = [f["n"] for f in g]
    minimo = [100 * (f["q_perm_min"] - 1) for f in g]
    maximo = [100 * (f["q_perm_max"] - 1) for f in g]
    mediana = [100 * (f["q_perm_mediana"] - 1) for f in g]
    cota = [100 * max(c["trabajos"]) / f["LB"] for c, f in zip(conjuntos_p2, g)]
    eje.fill_between(ns, minimo, maximo, color=AZUL_M[m], alpha=0.15, linewidth=0)
    eje.plot(ns, mediana, color=AZUL_M[m], marker=MARCA_M[m], markersize=6,
             label="m = %d: mediana de %d órdenes (banda: mín.–máx.)" % (m, PERMUTACIONES))
    eje.plot(ns, cota, color=AZUL_M[m], linestyle=(0, (3, 2)), linewidth=1.2)
eje.plot([], [], color=TINTA_TENUE, linestyle=(0, (3, 2)), linewidth=1.2,
         label="cota de Graham para cualquier orden: p_max / LB")
eje.set_xscale("log")
eje.set_yscale("log")
eje.set_xlabel("n (número de trabajos, escala log)")
eje.set_ylabel("exceso sobre LB  100·(C − LB)/LB  [%]")
eje.set_title("List Scheduling: el efecto del orden se desvanece cuando crece n/m", loc="left")
lpt_en_lb = sum(f["C_LPT"] == f["LB"] for f in filas_p2)
eje.annotate("LPT: exceso 0 %% (C = LB) en %d de %d casos,\n"
             "no representable en escala log" % (lpt_en_lb, len(filas_p2)),
             (0.99, 0.97), xycoords="axes fraction", ha="right", va="top", fontsize=8.5,
             color=TINTA_SECUNDARIA)
eje.legend(loc="lower left", fontsize=8.5)
guardar_figura(figura, "p2_efecto_orden_escala.png")

# ---- Grafico 3: tiempo de List Scheduling, LPT y del ordenamiento ----------
figura, ejes = plt.subplots(1, 3, figsize=(11.5, 3.9), sharey=True)
for eje, m in zip(ejes, M_P2):
    g = [f for f in filas_p2 if f["m"] == m]
    ns = [f["n"] for f in g]
    eje.plot(ns, [f["t_LS"] for f in g], color=COLOR["LS"], marker=MARCADOR["LS"],
             label="List Scheduling")
    eje.plot(ns, [f["t_LPT"] for f in g], color=COLOR["LPT"], marker=MARCADOR["LPT"],
             label="LPT (ordenar + LS)")
    eje.plot(ns, [f["t_orden"] for f in g], color=TINTA_TENUE, marker="^",
             linestyle=(0, (3, 2)), linewidth=1.4, label="solo ordenar")
    eje.set_xscale("log")
    eje.set_yscale("log")
    eje.set_title("m = %d" % m, loc="left")
    eje.set_xlabel("n (escala log)")
ejes[0].set_ylabel("tiempo mediano (s, escala log)")
ejes[0].legend(loc="upper left")
figura.suptitle("Tiempo de ejecución: ambos crecen casi linealmente; LPT paga además el "
                "ordenamiento O(n log n)", x=0.01, ha="left", fontsize=10,
                color=TINTA_SECUNDARIA)
figura.tight_layout()
guardar_figura(figura, "p2_tiempos.png")

### Pregunta de análisis: ¿el costo del ordenamiento de LPT es significativo?

In [ ]:
# ============================================================================
# Celda 14 - Propuesto 2: costo del ordenamiento frente a la mejora de LPT
# ============================================================================
# Pregunta: el costo de ordenar (O(n log n)) resulta significativo frente a la
# mejora del makespan? Se compara, para cada (n, m):
#   * sobrecosto de tiempo  = (t_LPT - t_LS) / t_LS
#   * peso del ordenamiento = t_ordenar / t_LPT
#   * mejora del makespan   = (mediana de LS con orden aleatorio - C_LPT) / mediana
# La mejora se mide contra la MEDIANA de los ordenes aleatorios, que representa
# lo que obtendria List Scheduling con un orden de llegada cualquiera.

filas_costo = []
for f in filas_p2:
    filas_costo.append([
        f["n"], f["m"], "%.1f" % (f["n"] / f["m"]),
        formato_tiempo(f["t_LS"]), formato_tiempo(f["t_LPT"]),
        formato_tiempo(f["t_LPT"] - f["t_LS"]),
        "%+.0f %%" % f["sobrecosto_LPT_pct"],
        "%.0f %%" % (100 * f["t_orden"] / f["t_LPT"]),
        "%g" % (f["C_perm_mediana"] - f["C_LPT"]),
        "%.3f %%" % f["mejora_LPT_pct"],
    ])
mostrar_tabla(["n", "m", "n/m", "t LS", "t LPT", "t extra", "sobrecosto",
               "ordenar / t LPT", "mejora (unid.)", "mejora %"], filas_costo,
              "Costo del ordenamiento frente a la mejora del makespan")
guardar_csv("propuesto2_costo_beneficio.csv",
            ["n", "m", "n_sobre_m", "t_LS", "t_LPT", "t_extra", "sobrecosto_pct",
             "peso_ordenamiento", "mejora_unidades", "mejora_pct"], filas_costo)

# Lectura automatica de los extremos para el informe.
mas_mejora = max(filas_p2, key=lambda f: f["mejora_LPT_pct"])
menos_mejora = min(filas_p2, key=lambda f: f["mejora_LPT_pct"])
print("\nMayor mejora: n=%d, m=%d -> %.1f %% del makespan por %s extra." % (
    mas_mejora["n"], mas_mejora["m"], mas_mejora["mejora_LPT_pct"],
    formato_tiempo(mas_mejora["t_LPT"] - mas_mejora["t_LS"])))
print("Menor mejora: n=%d, m=%d -> %.4f %% del makespan por %s extra." % (
    menos_mejora["n"], menos_mejora["m"], menos_mejora["mejora_LPT_pct"],
    formato_tiempo(menos_mejora["t_LPT"] - menos_mejora["t_LS"])))
print("Sobrecosto de LPT: entre %+.0f %% y %+.0f %% del tiempo de LS." % (
    min(f["sobrecosto_LPT_pct"] for f in filas_p2), max(f["sobrecosto_LPT_pct"] for f in filas_p2)))

**Lectura.** En términos relativos LPT tarda más que LS (hay que ordenar, $O(n \log n)$ frente
a $O(n \log m)$), pero en términos absolutos el costo es pequeño: milisegundos incluso con
$10^5$ trabajos. La mejora del makespan depende de $n/m$: es grande cuando hay pocos trabajos
por máquina y casi nula cuando hay muchos, porque entonces cualquier orden ya queda cerca
de $LB$ (cota $p_{\max}/LB$). Con estos tamaños el ordenamiento **no** es un costo
significativo: cuando LPT mejora mucho lo hace a cambio de microsegundos, y cuando mejora
poco tampoco cuesta casi nada. Solo sería relevante si la planificación se recalculara
miles de veces por segundo con $n/m$ grande, donde la ganancia es despreciable.

---
## 9. Descargar los resultados

Colab borra el disco de la sesión al cerrarla. Esta celda empaqueta los CSV, las instancias
en JSON y los gráficos en un ZIP y lo descarga.

In [ ]:
# ============================================================================
# Celda 15 - Empaquetar y descargar los resultados
# ============================================================================
# Colab borra el disco de la sesion al cerrarla: esta celda reune los CSV, las
# instancias (JSON) y los graficos (PNG) en un ZIP y lo descarga.
ruta_zip = shutil.make_archive(CARPETA_BASE + "_resultados", "zip", CARPETA_BASE)
print("ZIP generado: %s (%.1f KB)\n" % (ruta_zip, os.path.getsize(ruta_zip) / 1024))

for carpeta in (CARPETA_RESULTADOS, CARPETA_FIGURAS):
    print(carpeta)
    for nombre in sorted(os.listdir(carpeta)):
        print("   " + nombre)

if EN_COLAB:
    from google.colab import files
    files.download(ruta_zip)              # abre la descarga en el navegador
else:
    print("\nFuera de Colab: el ZIP quedo en la ruta indicada arriba.")

---
## Notas finales

- **Tiempos:** cambian en cada ejecución porque Colab asigna máquinas virtuales compartidas.
  Los contadores (makespans, cotas, nodos) son deterministas.
- **Reproducibilidad:** `resultados/instancias.json` conserva las instancias exactas del
  propuesto 1 y las semillas del propuesto 2.
- Para empezar de cero: `Entorno de ejecución → Reiniciar y ejecutar todo`.